<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png"  width='15%'>  

# RAG Evalulation Metrics



In [2]:
# !pip uninstall -y ragas langchain-community

Found existing installation: ragas 0.4.3
Uninstalling ragas-0.4.3:
  Successfully uninstalled ragas-0.4.3
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2


In [4]:
!pip install -q ragas "langchain-community<0.4.2" langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.6/123.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353.9/353.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [5]:
import ragas
print(ragas.__version__)

from ragas.llms import LangchainLLMWrapper
print("import succeeded")

0.4.3
import succeeded


In [6]:
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI
from langchain.chat_models import init_chat_model
from ragas.embeddings import OpenAIEmbeddings
import openai

import getpass
import os
from google.colab import userdata

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

evaluator_llm = LangchainLLMWrapper(init_chat_model("gpt-4o-mini", model_provider="openai"))
openai_client = openai.OpenAI()
evaluator_embeddings = OpenAIEmbeddings(client=openai_client)

/tmp/ipykernel_1644/2427267844.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(init_chat_model("gpt-4o-mini", model_provider="openai"))


# **Context Precision**

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_precision/


`Context Precision` is a metric that evaluates the retriever’s ability to rank relevant chunks higher than irrelevant ones for a given query in the retrieved context. Specifically, it assesses the degree to which relevant chunks in the retrieved context are placed at the top of the ranking.

It is calculated as the mean of the `precision@k` for each chunk in the context. `Precision@k` is the ratio of the number of relevant chunks at rank k to the total number of chunks at rank k.

$$
\text{Context Precision@K} =
\frac{\sum_{k=1}^{K} (\text{Precision@k} \times v_k)}
{\text{Total number of relevant items in the top } K \text{ results}}
$$

$$
\text{Precision@k} =
\frac{\text{true positives@k}}
{(\text{true positives@k} + \text{false positives@k})}
$$

Where $K$ is the total number of chunks in `retrieved_contexts` and
$v_k \in \{0, 1\}$ is the relevance indicator at rank $k$.




In [12]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."],
)


await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_12249/3278224137.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference


AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************PXQA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

Note that even if an irrelevant chunk is present at the second position in the array, context precision remains the same.



In [7]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris.", "The Brandenburg Gate is located in Berlin."],
)


await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_1644/1263925420.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference


0.9999999999

However, if this irrelevant chunk is placed at the first position, context precision reduces.



In [8]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Brandenburg Gate is located in Berlin.", "The Eiffel Tower is located in Paris." ],
)


await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_1644/2209227794.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference


0.49999999995

## 1. LLM Based Context Precision


### 1.1 Context Precision without reference
The `LLMContextPrecisionWithoutReference` metric can be used without the availability of a reference answer. To estimate if the retrieved contexts are relevant, this method uses the LLM to compare each chunk in retrieved_contexts with the response.

In [9]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."],
)


await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_1644/3278224137.py:2: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference


0.9999999999

### 1.2 Context Precision with reference

The `LLMContextPrecisionWithReference` metric can be used when you have both **retrieved contexts** and also a **reference response** associated with a user_input. To estimate if the retrieved contexts are relevant, this method uses the LLM to compare each chunk in retrieved_contexts with the reference.



In [10]:
from ragas import SingleTurnSample
from ragas.metrics import LLMContextPrecisionWithReference

context_precision = LLMContextPrecisionWithReference(llm=evaluator_llm)

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["The Eiffel Tower is located in Paris."],
)

await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_1644/326552884.py:2: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import LLMContextPrecisionWithReference


0.9999999999

## 2. Non LLM Based Context Precision

This metric uses non-LLM-based methods (such as Levenshtein distance measure) to determine whether a retrieved context is relevant.



### 2.1 Context Precision with reference contexts

The `NonLLMContextPrecisionWithReference` metric is designed for scenarios where both **retrieved contexts** and **reference contexts** are available for a user_input. To determine if a retrieved context is relevant, this method compares each retrieved context or chunk in retrieved_contexts with every context in reference_contexts using a non-LLM-based similarity measure.



In [11]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.6 MB/s eta 0:00:00


In [12]:
from ragas import SingleTurnSample
from ragas.metrics import NonLLMContextPrecisionWithReference

context_precision = NonLLMContextPrecisionWithReference()

sample = SingleTurnSample(
    retrieved_contexts=["The Eiffel Tower is located in Paris."],
    reference_contexts=["Paris is the capital of France.", "The Eiffel Tower is one of the most famous landmarks in Paris."]
)

await context_precision.single_turn_ascore(sample)

/tmp/ipykernel_1644/2556031695.py:2: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextPrecisionWithReference
  from ragas.metrics import NonLLMContextPrecisionWithReference


0.9999999999

In [ ]:
# help(SingleTurnSample)

# **Context Recall**

https://docs.ragas.io/en/stable/concepts/metrics/available_metrics/context_recall/#id-basedcontext-recall

Context Recall measures how many of the relevant documents (or pieces of information) were successfully retrieved. It focuses on not missing important results. Higher recall means fewer relevant documents were left out. In short, recall is about not missing anything important. Since it is about not missing anything, calculating context recall always requires a reference to compare against.



## 1. LLM Based Context Recall

`LLMContextRecall` is computed using `user_input`, `reference` and the `retrieved_contexts`, and the values range between 0 and 1, with higher values indicating better performance. This metric uses `reference` as a proxy to `reference_contexts` which also makes it easier to use as annotating `reference contexts` can be very time-consuming. To estimate `context recall` from the `reference`, the `reference` is broken down into claims each claim in the `reference answer` is analyzed to determine whether it can be attributed to the `retrieved context` or not. In an ideal scenario, all claims in the `reference answer` should be attributable to the `retrieved context`.

$$
\text{Context Recall} =
\frac{\text{Number of claims in the reference supported by the retrieved context}}
{\text{Total number of claims in the reference}}
$$




In [13]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import LLMContextRecall

sample = SingleTurnSample(
    user_input="Where is the Eiffel Tower located?",
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris.",
    retrieved_contexts=["Paris is the capital of France."],
)

context_recall = LLMContextRecall(llm=evaluator_llm)
await context_recall.single_turn_ascore(sample)

/tmp/ipykernel_1644/3318251018.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall


1.0

## 2. Non-LLM Based Context Recall

`NonLLMContextRecall` metric is computed using `retrieved_contexts` and `reference_contexts`, and the values range between 0 and 1, with higher values indicating better performance. This metrics uses non-LLM string comparison metrics to identify if a retrieved context is relevant or not. You can use any non LLM based metrics as distance measure to identify if a retrieved context is relevant or not.

$$
\text{context recall} =
\frac{|\text{Number of relevant contexts retrieved}|}
{|\text{Total number of reference contexts}|}
$$




In [14]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import NonLLMContextRecall

sample = SingleTurnSample(
    retrieved_contexts=["Paris is the capital of France."],
    reference_contexts=["Paris is the capital of France.", "The Eiffel Tower is one of the most famous landmarks in Paris."]
)

context_recall = NonLLMContextRecall()
await context_recall.single_turn_ascore(sample)

/tmp/ipykernel_1644/2355705596.py:2: DeprecationWarning: Importing NonLLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextRecall
  from ragas.metrics import NonLLMContextRecall


0.5

# **Context Entities Recall**


`ContextEntityRecall` metric gives the measure of recall of the `retrieved context`, based on the number of `entities` present in both `reference` and `retrieved_contexts` relative to the number of entities present in the reference alone. Simply put, it is a measure of what fraction of entities is recalled from reference. This metric is useful in fact-based use cases like tourism help desk, historical QA, etc. This metric can help evaluate the retrieval mechanism for entities, based on comparison with entities present in reference, because in cases where entities matter, we need the retrieved_contexts which cover them.


To compute this metric, we use two sets:

- *RE*: The set of entities in the reference.  
- *RCE*: The set of entities in the retrieved contexts.

$$
\text{Context Entity Recall} =
\frac{\text{Number of common entities between } RCE \text{ and } RE}
{\text{Total number of entities in } RE}
$$

In [17]:
from ragas import SingleTurnSample
from ragas.metrics import ContextEntityRecall

sample_1 = SingleTurnSample(
    reference="The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in the Indian city of Agra. It was commissioned in 1631 by the Mughal emperor Shah Jahan to house the tomb of his favorite wife, Mumtaz Mahal.",
    retrieved_contexts=["The Taj Mahal is a symbol of love and architectural marvel located in Agra, India. It was built by the Mughal emperor Shah Jahan in memory of his beloved wife, Mumtaz Mahal. The structure is renowned for its intricate marble work and beautiful gardens surrounding it"],
)

scorer = ContextEntityRecall(llm=evaluator_llm)

await scorer.single_turn_ascore(sample_1)


/tmp/ipykernel_1644/604796676.py:2: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextEntityRecall
  from ragas.metrics import ContextEntityRecall


0.6666666655555555

### Let's take a look at how this is actually calculated under the hood


- `reference`: The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in the Indian city of Agra. It was commissioned in 1631 by the Mughal emperor Shah Jahan to house the tomb of his favorite wife, Mumtaz Mahal.
- `High entity recall context`: The Taj Mahal is a symbol of love and architectural marvel located in Agra, India. It was built by the Mughal emperor Shah Jahan in memory of his beloved wife, Mumtaz Mahal. The structure is renowned for its intricate marble work and beautiful gardens surrounding it. - `Low entity recall context`: The Taj Mahal is an iconic monument in India. It is a UNESCO World Heritage Site and attracts millions of visitors annually. The intricate carvings and stunning architecture make it a must-visit destination.

In [18]:
reference="The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in the Indian city of Agra. It was commissioned in 1631 by the Mughal emperor Shah Jahan to house the tomb of his favorite wife, Mumtaz Mahal."
retrieved_contexts="The Taj Mahal is a symbol of love and architectural marvel located in Agra, India. It was built by the Mughal emperor Shah Jahan in memory of his beloved wife, Mumtaz Mahal. The structure is renowned for its intricate marble work and beautiful gardens surrounding it"

reference_entities = await scorer.get_entities(reference, callbacks=None)
retrieved_contexts_entities = await scorer.get_entities(retrieved_contexts, callbacks=None)

# Compare and inspect
print("Reference Entities:", reference_entities)
print("Retrieved Context Entities:", retrieved_contexts_entities)

Reference Entities: entities=['Taj Mahal', 'Yamuna', 'Agra', '1631', 'Shah Jahan', 'Mumtaz Mahal']
Retrieved Context Entities: entities=['Taj Mahal', 'Agra', 'India', 'Shah Jahan', 'Mumtaz Mahal']


In [19]:
# Compute overlap manually
common_entities = set(reference_entities.entities) & set(retrieved_contexts_entities.entities)
print("Common Entities:", common_entities)
print("Context Entity Recall =", len(common_entities) / len(reference_entities.entities))

Common Entities: {'Taj Mahal', 'Mumtaz Mahal', 'Agra', 'Shah Jahan'}
Context Entity Recall = 0.6666666666666666


### Another example with low score

In [20]:
sample_2 = SingleTurnSample(
    reference="The Taj Mahal is an ivory-white marble mausoleum on the right bank of the river Yamuna in the Indian city of Agra. It was commissioned in 1631 by the Mughal emperor Shah Jahan to house the tomb of his favorite wife, Mumtaz Mahal.",
    retrieved_contexts=["The Taj Mahal is an iconic monument in India. It is a UNESCO World Heritage Site and attracts millions of visitors annually. The intricate carvings and stunning architecture make it a must-visit destination."],
)

scorer = ContextEntityRecall(llm=evaluator_llm)

await scorer.single_turn_ascore(sample_2)

0.16666666638888888

# **Response Relevancy**

The `ResponseRelevancy` metric measures how relevant a response is to the user input. Higher scores indicate better alignment with the user input, while lower scores are given if the response is incomplete or includes redundant information.

This metric is calculated using the `user_input` and the `response` as follows:

1. Generate a set of artificial questions (default is 3) based on the response.  
   These questions are designed to reflect the content of the response.

2. Compute the cosine similarity between the embedding of the user input ($E_o$)  
   and the embedding of each generated question ($E_{g_i}$).

3. Take the average of these cosine similarity scores to get the **Answer Relevancy**:

$$
\text{Answer Relevancy} =
\frac{1}{N} \sum_{i=1}^{N}
\text{cosine similarity}(E_{g_i}, E_o)
$$

$$
\text{Answer Relevancy} =
\frac{1}{N} \sum_{i=1}^{N}
\frac{E_{g_i} \cdot E_o}{\|E_{g_i}\| \|E_o\|}
$$

Where:

- $E_{g_i}$: Embedding of the $i^{th}$ generated question.  
- $E_o$: Embedding of the user input.  
- $N$: Number of generated questions (default is 3).

**Note:** While the score usually falls between 0 and 1,  
it is not guaranteed due to cosine similarity's mathematical range of -1 to 1.

In [21]:
from ragas import SingleTurnSample
from ragas.metrics import ResponseRelevancy
from langchain_community.embeddings import HuggingFaceEmbeddings


sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )

evaluator_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
scorer = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
await scorer.single_turn_ascore(sample)

/tmp/ipykernel_1644/299380823.py:2: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import ResponseRelevancy
/tmp/ipykernel_1644/299380823.py:14: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  evaluator_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Ple

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

np.float64(0.956758769581393)

# **Faithfulness**

The `Faithfulness` metric measures how factually consistent a response is with the retrieved context. It ranges from 0 to 1, with higher scores indicating better consistency.

A response is considered faithful if all its claims can be supported by the retrieved context.

To calculate this:

1. Identify all the claims in the response.
2. Check each claim to see if it can be inferred from the retrieved context.
3. Compute the faithfulness score using the formula:

$$
\text{Faithfulness Score} =
\frac{\text{Number of claims in the response supported by the retrieved context}}
{\text{Total number of claims in the response}}
$$


In [22]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
        user_input="When was the first super bowl?",
        response="The first superbowl was held on Jan 15, 1967",
        retrieved_contexts=[
            "The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles."
        ]
    )
scorer = Faithfulness(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

/tmp/ipykernel_1644/803883764.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


1.0

In [23]:
type(scorer)

ragas.metrics._faithfulness.Faithfulness

In [24]:
sample

SingleTurnSample(user_input='When was the first super bowl?', retrieved_contexts=['The First AFL–NFL World Championship Game was an American football game played on January 15, 1967, at the Los Angeles Memorial Coliseum in Los Angeles.'], reference_contexts=None, retrieved_context_ids=None, reference_context_ids=None, response='The first superbowl was held on Jan 15, 1967', multi_responses=None, reference=None, rubrics=None, persona_name=None, query_style=None, query_length=None)

### How is it calculated?

**Question**: Where and when was Einstein born?

**Context**: Albert Einstein (born 14 March 1879) was a German-born theoretical physicist, widely held to be one of the greatest and most influential scientists of all time

**High faithfulness answer**: Einstein was born in Germany on 14th March 1879.

**Low faithfulness answer**: Einstein was born in Germany on 20th March 1879.


Let's examine how faithfulness was calculated using the low faithfulness answer:

- Step 1: Break the generated answer into individual statements.
  - Statements:
    - Statement 1: "Einstein was born in Germany."
    - Statement 2: "Einstein was born on 20th March 1879."
- Step 2: For each of the generated statements, verify if it can be inferred from the given context.
  - Statement 1: Yes
  - Statement 2: No
- Step 3: Use the formula depicted above to calculate faithfulness.
$$
\text{Faithfulness} = \frac{1}{2} = 0.5
$$



In [25]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import Faithfulness

sample = SingleTurnSample(
        user_input="Where and when was Einstein born?",
        response="Einstein was born in Germany on 20th March 1879.",
        retrieved_contexts=[
            "Albert Einstein (born 14 March 1879) was a German-born theoretical physicist, widely held to be one of the greatest and most influential scientists of all time"
        ]
    )
scorer = Faithfulness(llm=evaluator_llm)
await scorer.single_turn_ascore(sample)

/tmp/ipykernel_1644/1155672968.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


0.5

# **FactualCorrectness**

FactualCorrectness is a metric that compares and evaluates the factual accuracy of the generated response with the reference. This metric is used to determine the extent to which the generated response aligns with the reference. The factual correctness score ranges from 0 to 1, with higher values indicating better performance. To measure the alignment between the response and the reference, the metric uses the LLM to first break down the response and reference into claims and then uses natural language inference to determine the factual overlap between the response and the reference. Factual overlap is quantified using precision, recall, and F1 score, which can be controlled using the mode parameter.

The formula for calculating `True Positive` (TP), `False Positive` (FP), and `False Negative` (FN) is as follows:

- True Positive (TP) = Number of claims in response that are present in reference  

- False Positive (FP) = Number of claims in response that are not present in reference  

- False Negative (FN) = Number of claims in reference that are not present in response  

The formula for calculating precision, recall, and F1 score is as follows:

$$
\text{Precision} = \frac{TP}{(TP + FP)}
$$

$$
\text{Recall} = \frac{TP}{(TP + FN)}
$$

$$
\text{F1 Score} = \frac{2 \times \text{Precision} \times \text{Recall}}{(\text{Precision} + \text{Recall})}
$$


### Default metric is F1

In [26]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._factual_correctness import FactualCorrectness


sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. It has a height of 1000ft."
)

scorer = FactualCorrectness(llm = evaluator_llm)
await scorer.single_turn_ascore(sample)

np.float64(0.67)

In [27]:
sample.response

'The Eiffel Tower is located in Paris.'

In [28]:
sample.reference

'The Eiffel Tower is located in Paris. It has a height of 1000ft.'

In [29]:
scorer.claim_decomposition_prompt

ClaimDecompositionPrompt(instruction=
    Decompose and break down each of the input sentences into one or more standalone statements. Each statement should be a standalone claim that can be independently verified.
    Follow the level of atomicity and coverage as shown in the examples.
    , examples=[(ClaimDecompositionInput(response='Charles Babbage was a French mathematician, philosopher, and food critic.'), ClaimDecompositionOutput(claims=['Charles Babbage was a mathematician and philosopher.'])), (ClaimDecompositionInput(response='Albert Einstein was a German theoretical physicist. He developed the theory of relativity and also contributed to the development of quantum mechanics.'), ClaimDecompositionOutput(claims=['Albert Einstein was a German physicist.', 'Albert Einstein developed relativity and contributed to quantum mechanics.']))], language=english)

In [30]:
await scorer.decompose_and_verify_claims(reference=sample.response, response=sample.reference, callbacks=None)

array([ True, False])

In [31]:
await scorer.decompose_claims(response=sample.reference, callbacks=None)

['The Eiffel Tower is located in Paris.',
 'The Eiffel Tower has a height of 1000ft.']

### Use Precision

In [32]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._factual_correctness import FactualCorrectness


sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. It has a height of 1000ft."
)

scorer = FactualCorrectness(llm = evaluator_llm, mode="precision")
await scorer.single_turn_ascore(sample)

np.float64(1.0)

In [33]:
await scorer.decompose_claims(response=sample.reference, callbacks=None)

['The Eiffel Tower is located in Paris.',
 'The Eiffel Tower has a height of 1000ft.']

### Adjusting the `Atomicity` and `Coverage`

Each sentence in the response and reference can be broken down into one or more claims. The number of claims that are generated from a single sentence is determined by the level of `atomicity` and `coverage` required for your application.



In [34]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._factual_correctness import FactualCorrectness

reference = '''
  FactualCorrectness is a metric that compares and evaluates the factual accuracy of the
  generated response with the reference. This metric is used to determine the extent to
  which the generated response aligns with the reference. The factual correctness score
  ranges from 0 to 1, with higher values indicating better performance. To measure the
  alignment between the response and the reference, the metric uses the LLM to first break
  down the response and reference into claims and then uses natural language inference to
  determine the factual overlap between the response and the reference. Factual overlap is
  quantified using precision, recall, and F1 score, which can be controlled using the mode
  parameter.
'''
scorer = FactualCorrectness(llm = evaluator_llm, atomicity="high")
await scorer.decompose_claims(response=reference, callbacks=None)

['FactualCorrectness is a metric that compares and evaluates the factual accuracy of the generated response with the reference.',
 'FactualCorrectness is used to determine the extent to which the generated response aligns with the reference.',
 'The factual correctness score ranges from 0 to 1.',
 'Higher values of the factual correctness score indicate better performance.',
 'To measure the alignment between the response and the reference, the metric uses the LLM to break down the response and reference into claims.',
 'The metric uses natural language inference to determine the factual overlap between the response and the reference.',
 'Factual overlap is quantified using precision, recall, and F1 score.',
 'The precision, recall, and F1 score can be controlled using the mode parameter.']

In [35]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._factual_correctness import FactualCorrectness

reference = '''
  FactualCorrectness is a metric that compares and evaluates the factual accuracy of the
  generated response with the reference. This metric is used to determine the extent to
  which the ge
  nerated response aligns with the reference. The factual correctness score
  ranges from 0 to 1, with higher values indicating better performance. To measure the
  alignment between the response and the reference, the metric uses the LLM to first break
  down the response and reference into claims and then uses natural language inference to
  determine the factual overlap between the response and the reference. Factual overlap is
  quantified using precision, recall, and F1 score, which can be controlled using the mode
  parameter.
'''
scorer = FactualCorrectness(llm = evaluator_llm, atomicity="low")
await scorer.decompose_claims(response=reference, callbacks=None)

['FactualCorrectness is a metric that evaluates the factual accuracy of a generated response.',
 'FactualCorrectness compares the generated response with a reference.',
 'The factual correctness score ranges from 0 to 1.',
 'Higher factual correctness scores indicate better performance.',
 'The metric uses the LLM to break down the response and reference into claims.',
 'Natural language inference is used to determine the factual overlap between the response and the reference.',
 'Factual overlap is quantified using precision, recall, and F1 score.',
 'The mode parameter can control the quantification of factual overlap.']

# **Semantic Similarity**

The concept of Answer Semantic Similarity pertains to the assessment of the semantic resemblance between the generated answer and the ground truth. This evaluation is based on the ground truth and the answer, with values falling within the range of 0 to 1. A higher score signifies a better alignment between the generated answer and the ground truth.

Measuring the semantic similarity between answers can offer valuable insights into the quality of the generated response. This evaluation utilizes a bi-encoder model to calculate the semantic similarity score.



In [36]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import SemanticSimilarity
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings


evaluator_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample = SingleTurnSample(
    response="The Eiffel Tower is located in Paris.",
    reference="The Eiffel Tower is located in Paris. It has a height of 1000ft."
)

scorer = SemanticSimilarity(embeddings=LangchainEmbeddingsWrapper(evaluator_embeddings))
await scorer.single_turn_ascore(sample)

/tmp/ipykernel_1644/49216830.py:2: DeprecationWarning: Importing SemanticSimilarity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import SemanticSimilarity
  from ragas.metrics import SemanticSimilarity


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_1644/49216830.py:14: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  scorer = SemanticSimilarity(embeddings=LangchainEmbeddingsWrapper(evaluator_embeddings))


0.900936957016306